# Rough Implementation of Blueprint
 This notebook contains a rough implementation of the Blueprint. It is used to:
 - **Verify** if the concept works in reality
 - **Brainstorm** different **approaches** for each solution
- Identify **edge cases** in the solution.

Here in our demo we will use the **"Qwen 3.5 9B Q4 K M" (in GGUF format)** Open Source model as the central control brain.
And to use that LLM we will use **Llama.cpp (Python Library)**.

In [1]:
from llama_cpp import Llama, LlamaGrammar
# Llama grammar is required for forcing the output to be in our expected format.

We are **simulating** the industry server on our local computer.

Since on my **Mac** model is located in the downloads folder `/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf`

For real work it needs alot more context to breath, but for testing purpose I am keeping it small `2000`, so that it consumes less RAM.

In [4]:
qwen = Llama(
    model_path = "/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf",
    n_ctx = 2000,
    verbose = False # For agentic task we don't need verbose responses
)
print("Model Loaded Successfully!!")

output = qwen.create_chat_completion(
    messages=[{"role": "system", "content": "Hello How can I help you?"},
              {"role": "user", "content": "Hello"}],
    temperature=0.75,
    max_tokens=1000,
)
print(output)

Model Loaded Successfully!!
{'id': 'chatcmpl-a1f8bc8a-7f1f-4d97-8b48-c101cbbc2ecd', 'object': 'chat.completion', 'created': 1790162970, 'model': '/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Hello! How can I help you today? Feel free to ask me questions, need advice, or just want to chat.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 25, 'completion_tokens': 25, 'total_tokens': 50}}


For now we wil just use a sample json output of the Qwen model.

In [5]:
completion = output["choices"][0]["message"]["content"]
print(completion)

Hello! How can I help you today? Feel free to ask me questions, need advice, or just want to chat.


For now, we first need to brainstorm on the capabilities of the agent and design the execution of reach of the task.
Capabilities:
1. Read/Write Files
2. List files, can create directories
3. Fetch the meta data for each file.
4. Managing git like log and version control, where each commit is trackable.
5. Run command in the sandbox, code execution (Python or JavaScript or both)

We will store the relative path of the workspace of the agent in the `rel_path` variable.
We will use `pathlib` to deal with paths.

- `.resolve()` creates absolute path from relative path, with respect to working directory
- `Path()` converts the input into a path variable.
- `.mkdir(exist_ok = True, parents = True)` It creates directory at the given location, `exist_ok = True` ensures no error occurs if the directory already exists, `parents = True` creates any parent directory if missing instead of throwing errors.

In [13]:
from pathlib import Path
rel_path = Path('./workspace').resolve()
print(rel_path)
rel_path.mkdir(exist_ok = True, parents = True)

/Users/noyan/Desktop/Smart India Hackathon/AgenticAI/workspace
